In [1]:
"""
Summaries of the Wildfire Risk to Communities, Housing Unit Risk
Maxwell.Cook@colostate.edu
"""

import os, sys
import rasterio as rio

from rasterstats import zonal_stats

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Ready !


In [2]:
# load and prepare the incidents and perimeter data

# read in the incidents data (gathered from the ICS209+ by Evan Herpe et al)
# this subset is "timber only" fires
fp = os.path.join(projdir, "data/tabular/incidents/ics209plus-wf_incidents_2014_2022_Timber.csv")
incis = pd.read_csv(fp)
print(f"{len(incis)} incidents in the subset.")

# read in the fire data (MTBS)
perims = os.path.join(projdir, 'data/spatial/wf_incidents_2014_2022_mtbs_perims.gpkg')
perims = gpd.read_file(perims)
# clean the dataframe just a bit for processing
perims = perims[['Event_ID','geometry']]

# join to the incident subset
inci_perims = incis.merge(perims, on="Event_ID")
inci_perims = gpd.GeoDataFrame(
    inci_perims,
    geometry=inci_perims["geometry"],  # or use the correct column name for MTBS geometry
    crs=perims.crs  # use the CRS from the original MTBS GeoDataFrame
)

# save this file
out_fp = os.path.join(projdir, 'data/spatial/wf_incidents_2014_2022_perims_Timber.gpkg')
inci_perims.to_file(out_fp)
print(f"Saved to: {out_fp}")

# reproject to WGS
inci_perims = inci_perims.to_crs("EPSG:5070")
print(f"Matched [{len(inci_perims)}/{len(incis)}] with GIS perimeters")
del perims

1110 incidents in the subset.
Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/wf_incidents_2014_2022_perims_Timber.gpkg
Matched [1110/1110] with GIS perimeters


In [3]:
# load fire perimeters and intersect with counties
# calculate the intersecting counties and overlap area
counties = os.path.join(datadir,"boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp")
counties = gpd.read_file(counties).to_crs("EPSG:5070")

# Intersect with fire perimeters
fire_fips = gpd.overlay(inci_perims, counties[['GEOID', 'geometry']], how="intersection", keep_geom_type=True)
fire_fips['overlap_acres'] = fire_fips.geometry.area / 4046.86  # calculate the area in acres
# calculate the proportion of burned area in each county
fire_fips['county_prop'] = round(fire_fips['overlap_acres'] / fire_fips['BurnBndAc'], 2)
fire_fips[['Event_ID','GEOID','overlap_acres','county_prop']].head()

,Event_ID,GEOID,overlap_acres,county_prop
0,NC3492707684920220414,37049,719.301286,1.0
1,WA4896112149320220830,53073,2242.290778,1.0
2,AL3277908693520190402,01021,1206.773684,1.0
3,WA4888612140620220825,53073,3186.669262,1.0
4,CA3633311865120220803,06107,2266.921523,1.0


In [4]:
# load the WRC data downloaded from the USFS Research Data Archive
hurisk_fp = list_files(datadir, 'HURisk_CONUS.tif', recursive=True)[0]
popden_fp = list_files(datadir, 'PopDen_CONUS.tif', recursive=True)[0]

# find the counties overlapping fire data
counties_fire = counties[counties['GEOID'].isin(fire_fips['GEOID'].unique())]

# now calculate the mean HURisk and PopDen for each county:

# Load the HURisk raster and mask values > 0
with rio.open(hurisk_fp) as src:
    hurisk = src.read(1)        # read band 1
    transform = src.transform
    nodata = src.nodata

# Mask invalid or unwanted values
hurisk = hurisk.astype(float)
hurisk[hurisk == nodata] = np.nan
hurisk[hurisk <= 0] = np.nan  # keep only positive (nonzero) values

# Now pass to zonal_stats
county_stats = pd.DataFrame(zonal_stats(
    vectors=counties_fire,
    raster=hurisk,
    affine=transform,
    stats=['mean'],   # this will now calculate mean of >0 values
    nodata=np.nan     # make sure NaNs are excluded
))

county_stats['GEOID'] = counties_fire['GEOID'].values
county_stats = county_stats.rename(columns={'mean': 'WRC_HURisk'})

# Repeat for PopDen
popden_stats = pd.DataFrame(zonal_stats(
    counties_fire,
    popden_fp,
    stats=['mean'],
    geojson_out=False
))
popden_stats['GEOID'] = counties_fire['GEOID'].values
popden_stats = popden_stats.rename(columns={'mean': 'WRC_PopDen'})

# Join them together
county_summary = county_stats.merge(popden_stats, on='GEOID')

# save a county shapefile:
counties = counties.merge(county_summary, on='GEOID')
counties.to_file(os.path.join(projdir,'data/spatial/counties_hurisk_popden.gpkg'))

county_summary.head()

,WRC_HURisk,GEOID,WRC_PopDen
0,23816.520128,06091,1.446153
1,191.314229,21053,19.963024
2,1741.924384,01027,9.639502
3,2132.140363,41063,0.947596
4,182.778710,08109,0.834110


In [5]:
# join county-level risk/population summaries to the fire-county overlaps
fire_fips_c = fire_fips.merge(county_summary, on='GEOID', how='left')

# calculate the weighted average by fire
weighted_summary = (
    fire_fips_c
    .groupby('Event_ID')
    .apply(lambda g: pd.Series({
        'WRC_HURisk_wt': (g['WRC_HURisk'] * g['county_prop']).sum(),
        'WRC_PopDen_wt': (g['WRC_PopDen'] * g['county_prop']).sum()
    }), include_groups=False)
    .reset_index()
)

# check the results
weighted_summary.head()

,Event_ID,WRC_HURisk_wt,WRC_PopDen_wt
0,AL3267208704520220215,183.557614,5.348242
1,AL3277908693520190402,789.785916,27.134026
2,AL3311108808220210226,142.684736,9.126010
3,AL3326608609720161012,1741.924384,9.639502
4,AL3337208592120161029,1741.924384,9.639502


In [7]:
# save the table
out_fp = os.path.join(projdir,'data/tabular/WRC_hurisk_popden_weighted.csv')
weighted_summary.to_csv(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/WRC_hurisk_popden_weighted.csv
